In [2]:
#Basic chatbot with Memory , for that we will use checkpoint.memory 

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
#basemessage is collection of all HumanMessage,AIMessage
from typing import TypedDict, Annotated

from langgraph.checkpoint.memory import MemorySaver #****



In [3]:
from langgraph.graph import add_messages

class ChatState(TypedDict):
       messages:Annotated[list[BaseMessage], add_messages]

In [4]:
model=ChatOpenAI()

def chat_node(state:ChatState):
    #take user query from state
    messages=state['messages']
    #send to llm
    response=model.invoke(messages)
    #response store state
    return {'messages':[response]}

In [5]:
checkpointer=MemorySaver() #***  Add the checpointer , include this checkpointer in graph.compile()
#after this add the threadid( threadid is one interation with the chatbot)

graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)  #**** Add the MemorySaver checkpointer here
inital_state={
    'messages':[HumanMessage(content='What is the capital of India')]
}

chatbot.invoke(inital_state)




ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
thread_id='1'  #*** threadid for each one session conversation and now add the config dictionary

while True:
    user_message=input('TypeHere:')
    print("User:",user_message)

    if user_message.strip().lower() in ['quit','exit','bye']:
        break
    config={'configurable':{'thread_id':thread_id}}  #** adding the config for the threadid,pass this while passing message


    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)  #**

    print("AI Message: ",response['messages'][-1].content)

    #here we can see the chat history saved in the Memory saver 




User: Hi My name is Sachin
AI Message:  Hello Sachin, nice to meet you! How can I assist you today?
User: what is my name?
AI Message:  Your name is Sachin.
User: can you add 2, 2 
AI Message:  Sure! 2 + 2 equals 4.
User: can you multiply the result with 10
AI Message:  Of course! 4 multiplied by 10 is equal to 40.
User: exit
